# Introduction to Data Ingestion

In [1]:
!uv pip show langchain-text-splitters

Name: langchain-text-splitters
Version: 1.1.3
Location: D:\Mission Project\RAG\.venv\Lib\site-packages
Requires: langchain-core
Required-by: langchain-classic


Using Python 3.13.15 environment at: D:\Mission Project\RAG\.venv


In [2]:
import os
from typing import List, Dict, Any
import pandas as pd

In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("Setup is completed")

Setup is completed


## Understand Document Structure in Langchain

In [9]:
## Create a simple document
doc = Document (
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source":"example.txt",
        "page":1,
        "author": "Krish Naik",
        "date_created": "2024-01-01",
        "custom_field": "any_value"
    }
)
print("Document Structure")

print(f"Content : {doc.page_content}")
print(f"Metadata : {doc.metadata}")

# Why metadata matters:
print("\n Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Content : This is the main text content that will be embedded and searched.
Metadata : {'source': 'example.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01', 'custom_field': 'any_value'}

 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [10]:
type(doc)

langchain_core.documents.base.Document

### Text Files (.txt) - The Simplest Case

In [12]:
## Create a simple txt file
import os
os.makedirs("data/text_files", exist_ok = True)

In [13]:
sample_texts= {
    "data/text_files/python_intro.txt": """Python Programming Introduction
    
Python is a high-level, interpreted programming language that is easy to learn and used for building different types of applications.
Key Features of Python
1. Easy to Learn - Simple and readable syntax.
2. Interpreted - Code runs directly without separate compilation.
3. Dynamically Typed - Variable type declare karna zaroori nahi.
4. Object-Oriented - Classes and objects support karta hai.""",

"data/text_files/machine_learning.txt": """ Machine Learning Basics

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms to identify patterns in data and improve its performance with experience. Machine Learning is mainly divided into Supervised Learning, Unsupervised Learning, and Reinforcement Learning. It is widely used in recommendation systems, fraud detection, image recognition, speech recognition, self-driving cars, and medical diagnosis. For example, YouTube and Netflix use Machine Learning to recommend content according to user interests. ML helps organizations automate tasks, analyze large amounts of data, and make better decisions. With the rapid growth of data and computing power, Machine Learning has become an important technology in modern applications and is expected to play an even bigger role in the future.
Key Points of Machine Learning
- Learns from data
- Identifies patterns
- Makes predictions and decisions
- Improves with experience
"""
}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample file is created!")

Sample file is created!


### TextLoader - Read Single file

In [18]:

from langchain_community.document_loaders import TextLoader

##Loading a single text file
loader=TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(type(documents))
print(documents)

print(f"Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language that is easy to learn and used for building different types of applications.\nKey Features of Python\n1. Easy to Learn - Simple and readable syntax.\n2. Interpreted - Code runs directly without separate compilation.\n3. Dynamically Typed - Variable type declare karna zaroori nahi.\n4. Object-Oriented - Classes and objects support karta hai.')]
Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language that is ea...
Metadata: {'source': 'data/text_files/python_intro.txt'}


### DirectoryLoader - Multiple Text Files

In [19]:
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls = TextLoader, ## Loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

documents=dir_loader.load()

100%|██████████| 2/2 [00:00<00:00, 2423.75it/s]


In [20]:
print(f"Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f" Source: {doc.metadata['source']}")
    print(f" Length: {len(doc.page_content)} characters")

Loaded 2 documents

Document 1:
 Source: data\text_files\machine_learning.txt
 Length: 1087 characters

Document 2:
 Source: data\text_files\python_intro.txt
 Length: 427 characters


In [22]:
# Analysis
print("\n Directory Characteristics: ")
print("Advantages: ")
print(" -Loads multiple files at once")
print(" -Supports glob patterns")
print(" -Progress tracking")
print(" -Recursive directory scanning")

print(" \n Disadvantages: ")
print(" -All files must be same type")
print(" -Limited error handling per file")
print(" -Can be memory intensive for large directories")


 Directory Characteristics: 
Advantages: 
 -Loads multiple files at once
 -Supports glob patterns
 -Progress tracking
 -Recursive directory scanning
 
 Disadvantages: 
 -All files must be same type
 -Limited error handling per file
 -Can be memory intensive for large directories


### Text Splitting Statergies

In [43]:
### Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
print(documents)


[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content=' Machine Learning Basics\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms to identify patterns in data and improve its performance with experience. Machine Learning is mainly divided into Supervised Learning, Unsupervised Learning, and Reinforcement Learning. It is widely used in recommendation systems, fraud detection, image recognition, speech recognition, self-driving cars, and medical diagnosis. For example, YouTube and Netflix use Machine Learning to recommend content according to user interests. ML helps organizations automate tasks, analyze large amounts of data, and make better decisions. With the rapid growth of data and computing power, Machine Learning has become an important technology in modern applications and is expected 

In [25]:
## Method 1: Character Text Splitter
text = documents[0].page_content
text

' Machine Learning Basics\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms to identify patterns in data and improve its performance with experience. Machine Learning is mainly divided into Supervised Learning, Unsupervised Learning, and Reinforcement Learning. It is widely used in recommendation systems, fraud detection, image recognition, speech recognition, self-driving cars, and medical diagnosis. For example, YouTube and Netflix use Machine Learning to recommend content according to user interests. ML helps organizations automate tasks, analyze large amounts of data, and make better decisions. With the rapid growth of data and computing power, Machine Learning has become an important technology in modern applications and is expected to play an even bigger role in the future.\nKey Points of Machine Learning\n- Learns f

In [28]:
#Method 1: Character-based splitting
print("CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator="\n", # Split on new lines
    chunk_size=200, # Max chunk size in characters
    chunk_overlap = 20, # Overlap between chunks
    length_function = len # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Created a chunk of size 927, which is longer than the specified 200


CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: Machine Learning Basics...


In [33]:
print(char_chunks[0])
print("--------------------")
print(char_chunks[1])
print("--------------------")
print(char_chunks[2])

Machine Learning Basics
--------------------
Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It uses algorithms to identify patterns in data and improve its performance with experience. Machine Learning is mainly divided into Supervised Learning, Unsupervised Learning, and Reinforcement Learning. It is widely used in recommendation systems, fraud detection, image recognition, speech recognition, self-driving cars, and medical diagnosis. For example, YouTube and Netflix use Machine Learning to recommend content according to user interests. ML helps organizations automate tasks, analyze large amounts of data, and make better decisions. With the rapid growth of data and computing power, Machine Learning has become an important technology in modern applications and is expected to play an even bigger role in the future.
--------------------
Key Po

In [37]:
# Method 2: Recursive Character splitting (RECOMMAMAMAMND)
print("\n RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    #try these seperators in order
    separators=["\n\n", "\n", " ", ""],
    chunk_size = 200,
    chunk_overlap= 20,
    length_function=len
)
recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0] [:100]}...")


 RECURSIVE CHARACTER TEXT SPLITTER
Created 8 chunks
First chunk: Machine Learning Basics...


In [38]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("-----------------")
print(recursive_chunks[2])
print("-----------------")
print(recursive_chunks[3])
print("-----------------")
print(recursive_chunks[4])
print("-----------------")

Machine Learning Basics
-----------------
Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and make predictions or decisions without being explicitly programmed for every task. It
-----------------
for every task. It uses algorithms to identify patterns in data and improve its performance with experience. Machine Learning is mainly divided into Supervised Learning, Unsupervised Learning, and
-----------------
Learning, and Reinforcement Learning. It is widely used in recommendation systems, fraud detection, image recognition, speech recognition, self-driving cars, and medical diagnosis. For example,
-----------------
For example, YouTube and Netflix use Machine Learning to recommend content according to user interests. ML helps organizations automate tasks, analyze large amounts of data, and make better
-----------------


In [40]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "], #Only split on spaces
    chunk_size= 80,
    chunk_overlap=20,
    length_function = len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")

    print()



Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'
Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'
Chunk 4: 'is sentence five. This is sentence six.'



In [41]:
# Method 3: Token-based Splitting
token_splitter = TokenTextSplitter(
    chunk_size=50, #Size in tokens (not characters)
    chunk_overlap = 10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")

Created 5 chunks
First chunk:  Machine Learning Basics

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that ena...


In [42]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")
print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  Use when: Working with token-limited models
